# llminfer All Examples (Colab)

This notebook walks through every example path in `examples/` with clear prints,
artifacts, and plots.

Covered:
- `full_demo.py` equivalent flow,
- `advanced_features.py` equivalent flow,
- `benchmark_vs_vllm.py` equivalent flow,
- `run_server.py` + `openai_api_client.py` equivalent flow.



## 0) Runtime
In Colab, choose **Runtime -> Change runtime type -> GPU** before running.


In [ ]:
!nvidia-smi

import os
import platform
import torch

print('Python :', platform.python_version())
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
else:
    print('Warning: CUDA is not enabled. Use a GPU runtime for realistic results.')


## 1) Clone + install

In [ ]:
REPO_URL = 'https://github.com/nickforce989/llminfer.git'
TARGET_DIR = '/content/llminfer'

import os
import shutil

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

!git clone -q {REPO_URL} {TARGET_DIR}
%cd {TARGET_DIR}

!pip -q install -U pip
!pip -q install -e ".[serve,peft]"


## 2) Helpers + run flags

In [ ]:
from pathlib import Path
from IPython.display import Image, display

RUN_VLLM = False          # set True to run vLLM sections
RUN_SERVING = True        # set False to skip API server sections
MODEL_FAST = 'facebook/opt-125m'
MODEL_ADV = 'Qwen/Qwen2.5-1.5B-Instruct'


def header(title: str):
    print('\n' + '=' * 20 + f' {title} ' + '=' * 20)


def show_image(path: str):
    p = Path(path)
    if p.exists():
        print(f'[plot] {p}')
        display(Image(filename=str(p)))
    else:
        print(f'[missing plot] {p}')


## 3) Example map

In [ ]:
import os
for name in sorted(os.listdir('examples')):
    if name.endswith('.py') or name.endswith('.ipynb'):
        print('-', name)


## 4) `full_demo.py` equivalent (trimmed for Colab speed)

In [ ]:
from llminfer import InferenceEngine, EngineConfig, Benchmarker
from llminfer.benchmark import BackendComparison
from llminfer.config import Backend

engine = InferenceEngine(EngineConfig(model_name=MODEL_FAST, max_batch_size=8, batch_timeout_ms=20))
engine.load()

header('1) basic run')
r = engine.run('What is a transformer neural network?', max_new_tokens=64, temperature=0.2)
print(r.generated_text)
print('Stats:', r.stats)

header('2) streaming')
for ch in engine.stream('Once upon a time in a GPU cluster,', max_new_tokens=48, temperature=0.2):
    if not ch.is_final:
        print(ch.token, end='', flush=True)
    else:
        print('\nFinal stats:', ch.stats)

header('3) batch')
prompts = [
    'What is gradient descent?',
    'What is batch normalization?',
    'What is FlashAttention?',
    'What is LoRA?',
]
for i, out in enumerate(engine.run_batch(prompts, max_new_tokens=40, temperature=0.2), 1):
    print(f'{i}. {out.generated_text[:120]}')

header('4) benchmark + compare')
bm = Benchmarker(engine)
b = bm.run(batch_sizes=[1, 2, 4], num_runs=2, warmup_runs=1, max_new_tokens=32, use_continuous_batching=True)
b.print_summary()

outdir = Path('benchmarks/all_examples/full_demo')
outdir.mkdir(parents=True, exist_ok=True)
b.plot_suite(str(outdir), prefix='full_demo')
show_image(str(outdir / 'full_demo_dashboard.png'))

cmp = BackendComparison(model_name=MODEL_FAST, backends=[Backend.EAGER, Backend.COMPILED], compile_cudagraphs=True)
cmp_res = cmp.run(batch_sizes=[1, 2, 4], num_runs=2, max_new_tokens=32, use_continuous_batching=True)
cmp.print_table(cmp_res)
cmp.plot(cmp_res, str(outdir / 'full_demo_compare.png'))
show_image(str(outdir / 'full_demo_compare.png'))


## 5) `advanced_features.py` equivalent

In [ ]:
from llminfer.config import Backend

adv = InferenceEngine(
    EngineConfig(
        model_name=MODEL_ADV,
        backend=Backend.EAGER,
        assistant_model_name=None,
        speculative_num_assistant_tokens=8,
        speculative_confidence_threshold=0.4,
        max_batch_size=8,
        batch_timeout_ms=20,
    )
)
adv.load()

header('Advanced constrained decode')
adv_res = adv.run(
    'Write two concise bullet points about GPUs for deep learning.',
    max_new_tokens=80,
    temperature=0.2,
    no_repeat_ngram_size=3,
    bad_words=['joking', 'not sure'],
    stop_sequences=['\n\n'],
    seed=7,
)
print(adv_res.generated_text)
print('Stats:', adv_res.stats)

header('Advanced artifact export')
adv_dir = Path('benchmarks/all_examples/advanced')
adv_dir.mkdir(parents=True, exist_ok=True)
adv_bench = Benchmarker(adv).run(batch_sizes=[1, 2, 4], num_runs=2, warmup_runs=1, max_new_tokens=40, use_continuous_batching=True)
adv_bench.save_json(str(adv_dir / 'benchmark.json'))
adv_bench.save_csv(str(adv_dir / 'benchmark.csv'))
adv_bench.plot_suite(str(adv_dir), prefix='advanced')
show_image(str(adv_dir / 'advanced_dashboard.png'))


## 6) `benchmark_vs_vllm.py` equivalent

In [ ]:
from llminfer.config import QuantMode

if RUN_VLLM:
    import platform

    print('Python:', platform.python_version())
    try:
        import torch
        print('Torch before install:', torch.__version__, 'CUDA:', torch.version.cuda)
    except Exception:
        print('Torch not importable before install')

    !pip -q uninstall -y vllm
    !pip -q install --upgrade --force-reinstall --no-cache-dir vllm
    !pip -q install jedi "protobuf<6"

    try:
        import vllm  # noqa: F401
        import torch
        print('vLLM import: OK')
        print('Torch after install:', torch.__version__, 'CUDA:', torch.version.cuda)
    except Exception as exc:
        print('[error] vLLM import failed:', exc)
        print('[hint] Try restarting runtime and rerunning from the top.')
        raise

    from llminfer.config import Backend

    cmp_v = BackendComparison(
        model_name=MODEL_FAST,
        backends=[Backend.EAGER, Backend.VLLM],
        quant_mode=QuantMode.NONE,
        tensor_parallel_size=1,
        pipeline_parallel_size=1,
        paged_kv=True,
        page_size_tokens=16,
    )
    rv = cmp_v.run(batch_sizes=[1, 2, 4], num_runs=2, max_new_tokens=40, use_continuous_batching=True)
    cmp_v.print_table(rv)

    vdir = Path('benchmarks/all_examples/vllm')
    vdir.mkdir(parents=True, exist_ok=True)
    cmp_v.plot(rv, str(vdir / 'comparison.png'))
    show_image(str(vdir / 'comparison.png'))
else:
    print('Skipping vLLM section. Set RUN_VLLM=True to enable.')


## 7) `run_server.py` + `openai_api_client.py` equivalent

In [ ]:
import json
import os
import signal
import subprocess
import time
import requests

if RUN_SERVING:
    host, port = '127.0.0.1', 8000
    if 'srv' in globals() and srv and srv.poll() is None:
        os.kill(srv.pid, signal.SIGTERM)
        time.sleep(1)

    srv_cmd = [
        'python', '-m', 'llminfer.cli', 'serve',
        '--model', MODEL_FAST,
        '--backend', 'eager',
        '--host', host,
        '--port', str(port),
        '--max-batch-size', '16',
        '--batch-timeout-ms', '20',
    ]
    srv = subprocess.Popen(srv_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    deadline = time.time() + 180
    while time.time() < deadline:
        if srv.poll() is not None:
            raise RuntimeError('Server exited early')
        try:
            hz = requests.get(f'http://{host}:{port}/healthz', timeout=2)
            if hz.ok:
                print('Server ready:', hz.json())
                break
        except Exception:
            pass
        time.sleep(1)

    payload = {
        'model': 'local-llminfer',
        'messages': [
            {'role': 'system', 'content': 'You are concise.'},
            {'role': 'user', 'content': 'Tell me about GPUs in 4 bullet points.'},
        ],
        'max_tokens': 120,
        'temperature': 0.2,
    }
    r = requests.post(f'http://{host}:{port}/v1/chat/completions', json=payload, timeout=180)
    r.raise_for_status()
    print('\nNon-streaming response:\n')
    print(r.json()['choices'][0]['message']['content'])

    print('\nStreaming response:\n')
    stream_payload = {
        'model': 'local-llminfer',
        'messages': [{'role': 'user', 'content': 'Explain KV cache in under 80 words.'}],
        'max_tokens': 120,
        'temperature': 0.2,
        'stream': True,
    }
    decoder = json.JSONDecoder()
    done = False
    with requests.post(f'http://{host}:{port}/v1/chat/completions', json=stream_payload, stream=True, timeout=180) as rs:
        rs.raise_for_status()
        for raw_line in rs.iter_lines(decode_unicode=True):
            if not raw_line:
                continue
            line = raw_line.strip()
            if 'data:' not in line:
                continue
            payloads = [p.strip() for p in line.split('data:') if p.strip()]
            for payload in payloads:
                if payload == '[DONE]':
                    done = True
                    break
                if payload in {'\\n', '\\n\\n', '"\\n"', '"\\n\\n"'}:
                    continue
                s = payload
                while s:
                    s = s.lstrip()
                    if not s:
                        break
                    try:
                        obj, idx = decoder.raw_decode(s)
                    except json.JSONDecodeError:
                        break
                    s = s[idx:]
                    delta = obj.get('choices', [{}])[0].get('delta', {}).get('content', '')
                    if delta:
                        print(delta, end='', flush=True)
            if done:
                print('\n[done]')
                break
else:
    print('Serving section skipped. Set RUN_SERVING=True to run.')


## 8) Summary artifacts

In [ ]:
import os
for p, _, files in os.walk('benchmarks/all_examples'):
    pngs = sorted([f for f in files if f.endswith('.png')])
    if pngs:
        print('\n', p)
        for f in pngs:
            print(' -', f)


## 9) Cleanup

In [ ]:
for obj_name in ['engine', 'adv']:
    if obj_name in globals():
        try:
            globals()[obj_name].unload()
        except Exception:
            pass

if 'srv' in globals() and srv and srv.poll() is None:
    import os, signal, time
    os.kill(srv.pid, signal.SIGTERM)
    time.sleep(1)

print('Done.')
